# 21. Fragrantica Note Canonicalization Audit

## 1. 분석 목적과 경계

Fragrantica 내부 Note 2,523개에서 동일한 향 개념이 표기·철자·문법 또는 기존 공식 alias 때문에 여러 raw Note로 분리됐는지 확인한다.

이번 Notebook은 다음 원칙만 사용한다.

- 기존 Note vocabulary와 Stage 18~20의 v0.3 dictionary/evidence를 재사용한다.
- 문자열 유사도는 후보 생성에만 사용한다.
- 확실한 표기 규칙 또는 기존 HIGH SAME_CONCEPT만 통합한다.
- FAMILY / RELATED / 근거 부족 후보는 통합하지 않는다.
- perfumes.csv, perfumes.jsonl, 외부 웹, LLM, embedding, ranking은 사용하지 않는다.
- 기존 dictionary/evidence와 Golden Set은 수정하지 않는다.
- 새 파일은 canonical map CSV와 summary Markdown만 생성한다.


In [1]:
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import json
import re
import unicodedata

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
PATHS = {
    "vocabulary": ROOT / "analysis_outputs" / "17_fragrantica_note_vocabulary.csv",
    "stage17_matching": ROOT / "analysis_outputs" / "17_note_ifra_matching.csv",
    "dictionary": ROOT / "data" / "scent_knowledge" / "scent_term_dictionary_v0.3.csv",
    "evidence": ROOT / "data" / "scent_knowledge" / "scent_term_evidence_v0.3.csv",
    "stage20_decisions": ROOT / "analysis_outputs" / "20_external_note_validation_decisions.csv",
    "golden_set": ROOT / "evaluation_data" / "stage1" / "13_stage1_golden_set_v1_200.xlsx",
}
MAP_PATH = ROOT / "data" / "scent_knowledge" / "fragrantica_note_canonical_map_v1.csv"
SUMMARY_PATH = ROOT / "analysis_outputs" / "21_fragrantica_note_canonicalization_summary.md"

for label, path in PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label}: {path}")

vocab = pd.read_csv(PATHS["vocabulary"], keep_default_na=False)
stage17 = pd.read_csv(PATHS["stage17_matching"], keep_default_na=False)
dictionary = pd.read_csv(PATHS["dictionary"], keep_default_na=False)
evidence = pd.read_csv(PATHS["evidence"], keep_default_na=False)
stage20_decisions = pd.read_csv(PATHS["stage20_decisions"], keep_default_na=False)

assert vocab.columns.tolist() == ["note", "perfume_count", "occurrence_count"]
assert len(vocab) == 2523
assert vocab["note"].nunique() == 2523
assert {"term", "canonical_term", "relation_type", "confidence"}.issubset(dictionary.columns)
assert {"term", "source_org", "supports_relation"}.issubset(evidence.columns)
assert {"note", "relation_type", "decision_reason"}.issubset(stage20_decisions.columns)

input_rows = [
    {"input": "vocabulary", "file": PATHS["vocabulary"].name, "rows": len(vocab)},
    {"input": "stage17_matching", "file": PATHS["stage17_matching"].name, "rows": len(stage17)},
    {"input": "dictionary", "file": PATHS["dictionary"].name, "rows": len(dictionary)},
    {"input": "evidence", "file": PATHS["evidence"].name, "rows": len(evidence)},
    {"input": "stage20_decisions", "file": PATHS["stage20_decisions"].name, "rows": len(stage20_decisions)},
    {"input": "golden_set", "file": PATHS["golden_set"].name, "rows": 200},
]
display(pd.DataFrame(input_rows))
print("perfumes.csv / perfumes.jsonl 재사용: 0건")


,input,file,rows
0,vocabulary,17_fragrantica_note_vocabulary.csv,2523
1,stage17_matching,17_note_ifra_matching.csv,2523
2,dictionary,scent_term_dictionary_v0.3.csv,197
3,evidence,scent_term_evidence_v0.3.csv,213
4,stage20_decisions,20_external_note_validation_decisions.csv,30
5,golden_set,13_stage1_golden_set_v1_200.xlsx,200


perfumes.csv / perfumes.jsonl 재사용: 0건


## 2. 후보 탐색 방식

전체 2,523개 Note에 다음 candidate key를 적용한다.

1. casefold + 공백/hyphen/punctuation 제거
2. terminal note/notes 제거
3. 보수적인 singular/plural key
4. 정규화 문자열 길이 차이 2 이하, SequenceMatcher 0.94 이상
5. v0.3 SAME_CONCEPT와 명시적으로 요청된 Vanilla/Vanille 후보

Candidate는 자동 정답이 아니다. 최종 판정은 별도의 명시적 decision table로 제한한다.


In [2]:
def normalized_words(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = re.sub(r"[-_/]+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def tight_key(text):
    return re.sub(r"[^a-z0-9]", "", normalized_words(text))


def without_note_suffix(text):
    tokens = normalized_words(text).split()
    if tokens and tokens[-1] in {"note", "notes"}:
        tokens = tokens[:-1]
    return " ".join(tokens)


def singular_token(token):
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 4 and token.endswith("es") and not token.endswith(("sses", "uses")):
        return token[:-2]
    if len(token) > 3 and token.endswith("s") and not token.endswith(("ss", "us", "is")):
        return token[:-1]
    return token


def morphology_key(text):
    tokens = without_note_suffix(text).split()
    if tokens:
        tokens[-1] = singular_token(tokens[-1])
    return " ".join(tokens)


notes = vocab["note"].tolist()
candidate_rows = []
seen_pairs = set()


def add_candidate(left, right, candidate_type, similarity=None):
    if left == right:
        return
    pair = tuple(sorted((left, right)))
    key = (pair, candidate_type)
    if key in seen_pairs:
        return
    seen_pairs.add(key)
    candidate_rows.append({
        "left_note": pair[0],
        "right_note": pair[1],
        "candidate_type": candidate_type,
        "similarity": similarity,
    })


for candidate_type, key_function in [
    ("CASE_SPACE_HYPHEN_PUNCTUATION", tight_key),
    ("NOTE_SUFFIX", without_note_suffix),
    ("SINGULAR_PLURAL", morphology_key),
]:
    groups = defaultdict(list)
    for note in notes:
        groups[key_function(note)].append(note)
    for group in groups.values():
        if len(group) > 1:
            for index, left in enumerate(group):
                for right in group[index + 1:]:
                    add_candidate(left, right, candidate_type)

tight = {note: tight_key(note) for note in notes}
for index, left in enumerate(notes):
    left_key = tight[left]
    if len(left_key) < 5:
        continue
    for right in notes[index + 1:]:
        right_key = tight[right]
        if (
            len(right_key) < 5
            or left_key == right_key
            or left_key[0] != right_key[0]
            or abs(len(left_key) - len(right_key)) > 2
        ):
            continue
        similarity = SequenceMatcher(None, left_key, right_key).ratio()
        if similarity >= 0.94:
            add_candidate(left, right, "STRING_SIMILARITY", round(similarity, 4))

for row in dictionary.loc[dictionary["relation_type"].eq("SAME_CONCEPT")].itertuples(index=False):
    if row.term in notes:
        add_candidate(row.term, row.canonical_term, "V0.3_SAME_CONCEPT")

add_candidate("Vanilla", "Vanille", "EXPLICIT_LANGUAGE_VARIANT")

candidates = pd.DataFrame(candidate_rows)
display(candidates.groupby("candidate_type").size().to_frame("candidate_pair_count"))
display(candidates.sort_values(["candidate_type", "left_note", "right_note"]).head(40))


,candidate_pair_count
candidate_type,
CASE_SPACE_HYPHEN_PUNCTUATION,15
EXPLICIT_LANGUAGE_VARIANT,1
NOTE_SUFFIX,10
SINGULAR_PLURAL,16
STRING_SIMILARITY,15
V0.3_SAME_CONCEPT,24


,left_note,right_note,candidate_type,similarity
7,AMBROX® SUPER,Ambrox Super,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
12,Angel's Trumpet,Angels Trumpet,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
3,Black Currant,Blackcurrant,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
8,Candy Apple,Candy apple,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
10,Cashmir wood,Cashmirwood,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
1,Lily of the Valley,Lily-of-the-Valley,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
6,Lime (Linden Blossom),Lime (Linden) Blossom,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
0,Oakmoss,oak moss,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
5,Passion Fruit,Passionfruit,CASE_SPACE_HYPHEN_PUNCTUATION,NaN
4,Plum,plum,CASE_SPACE_HYPHEN_PUNCTUATION,NaN


## 3. 판정 정책

- v0.3 HIGH SAME_CONCEPT는 그대로 재사용한다.
- case/space/hyphen/punctuation만 다른 명확한 표현과 제한된 철자 오류는 SAME_CONCEPT로 판정한다.
- 명백한 singular/plural 중 기존 raw 대표 용어를 안전하게 선택할 수 있는 경우만 통합한다.
- 기존 Evidence가 FAMILY/RELATED이거나 명시적으로 금지된 관계는 NOT_SAME으로 유지한다.
- 언어·지역 형용사·모호한 문법 차이는 REVIEW로 남긴다.


In [3]:
# v0.3 외에 이번 내부 vocabulary audit에서 확정하는 최소 목록이다.
CURATED_SAME = {
    # case / spacing / hyphen / punctuation
    "oak moss": "Oakmoss",
    "Ylang Ylang": "Ylang-Ylang",
    "plum": "Plum",
    "Passionfruit": "Passion Fruit",
    "Lime (Linden Blossom)": "Lime (Linden) Blossom",
    "AMBROX® SUPER": "Ambrox Super",
    "Candy apple": "Candy Apple",
    "Raspberry leaf": "Raspberry Leaf",
    "Cashmir wood": "Cashmirwood",
    "Sea Shells": "Seashells",
    "St John's Wort": "St. John's Wort",
    "Angels Trumpet": "Angel's Trumpet",
    "Quandong Desert Peach": "Quandong, desert peach",
    # clear spelling variants
    "Marshamallow": "Marshmallow",
    "Sandalowood": "Sandalwood",
    "Pitosporum": "Pittosporum",
    "Coton candy": "Cotton Candy",
    "Panacotta": "Panna Cotta",
    "Narciussus": "Narcissus",
    "Ethyl Vanilin": "Ethylvanillin",
    # clear singular / plural
    "juniper berry": "Juniper Berries",
    "Resins": "Resin",
    "Tropical Fruits": "Tropical Fruit",
    "Cereals": "Cereal",
}

REVIEW_DECISIONS = {
    "Vanille": "Vanilla/Vanille is a plausible language variant, but existing v0.3 evidence explicitly left Vanille unresolved.",
    "Californian Orange": "California/Californian wording is plausible but is not a pure spelling rule.",
    "Virginian Cedar": "Virginia/Virginian wording is plausible but existing evidence does not confirm identity.",
    "Woodsy Notes": "Existing v0.3 evidence explicitly did not equate Woodsy Notes with Woody.",
    "White Wood": "White Wood/White Woods may be grammatical variants, but existing evidence is unresolved.",
    "Cep": "Cep/Cepes may be a language/plural variant, but existing evidence is absent.",
    "Gaiac Wood": "Gaiac/Guaiac may be a spelling or language variant, but existing project evidence does not explicitly confirm alias identity.",
}

NOT_SAME_DECISIONS = {
    "Peppermint": "Mint is broader and existing v0.3 evidence separates Mentha material forms.",
    "Moss": "Moss is RELATED to Mossy and must not be collapsed with Oakmoss.",
    "Neroli": "Orange Blossom and Neroli are distinct bitter-orange flower materials in Stage 20 evidence.",
    "Pine needles": "Pine and Pine Needle material are not automatically identical concepts.",
    "White Musk": "Existing v0.3 relation to Musk-Like is FAMILY, not SAME_CONCEPT.",
    "Water Notes": "Existing v0.3 relation to Watery is RELATED, not SAME_CONCEPT.",
    "Bearberry": "Barberry and Bearberry are only string-similar and are different labels.",
}

vocabulary_set = set(notes)
for raw_note, canonical_note in CURATED_SAME.items():
    assert raw_note in vocabulary_set, raw_note
    assert canonical_note in vocabulary_set, canonical_note
for raw_note in set(REVIEW_DECISIONS) | set(NOT_SAME_DECISIONS):
    assert raw_note in vocabulary_set, raw_note

print("Curated SAME_CONCEPT:", len(CURATED_SAME))
print("REVIEW:", len(REVIEW_DECISIONS))
print("NOT_SAME:", len(NOT_SAME_DECISIONS))


Curated SAME_CONCEPT: 24
REVIEW: 7
NOT_SAME: 7


## 4. 전체 raw Note → canonical Note lookup 생성

Semantic Bridge와 검색에서 바로 사용할 수 있도록 2,523개 전체 mapping을 만든다. 변경 없는 Note는 CANONICAL이다.


In [4]:
mapping_by_note = {
    note: {
        "raw_note": note,
        "canonical_note": note,
        "relation": "CANONICAL",
        "evidence_source": "17_fragrantica_note_vocabulary.csv",
        "reason": "No confirmed duplicate expression; retained as its own canonical concept.",
        "manual_review_required": False,
    }
    for note in notes
}

same_v03 = (
    dictionary.loc[dictionary["relation_type"].eq("SAME_CONCEPT")]
    .drop_duplicates("term", keep="last")
)

for row in same_v03.itertuples(index=False):
    if row.term not in mapping_by_note:
        continue
    source_orgs = sorted(set(
        evidence.loc[
            evidence["term"].eq(row.term)
            & evidence["supports_relation"].eq("SAME_CONCEPT"),
            "source_org",
        ]
    ) - {""})
    source = "scent_term_dictionary_v0.3.csv; scent_term_evidence_v0.3.csv"
    if source_orgs:
        source += " (" + ", ".join(source_orgs) + ")"
    mapping_by_note[row.term].update({
        "canonical_note": row.canonical_term,
        "relation": "CANONICAL" if row.term == row.canonical_term else "SAME_CONCEPT",
        "evidence_source": source,
        "reason": "Existing HIGH-confidence SAME_CONCEPT decision reused without reinterpretation.",
        "manual_review_required": False,
    })

spelling_raw = {
    "Marshamallow", "Sandalowood", "Pitosporum", "Coton candy",
    "Panacotta", "Narciussus", "Ethyl Vanilin",
}
grammar_raw = {"juniper berry", "Resins", "Tropical Fruits", "Cereals"}

for raw_note, canonical_note in CURATED_SAME.items():
    if mapping_by_note[raw_note]["relation"] == "SAME_CONCEPT":
        continue
    if raw_note in spelling_raw:
        rule = "clear spelling"
    elif raw_note in grammar_raw:
        rule = "clear singular/plural"
    else:
        rule = "case/space/hyphen/punctuation"
    mapping_by_note[raw_note].update({
        "canonical_note": canonical_note,
        "relation": "SAME_CONCEPT",
        "evidence_source": f"Fragrantica internal vocabulary; {rule} rule",
        "reason": f"Existing raw labels differ only by a {rule} form; an existing representative raw label is retained.",
        "manual_review_required": False,
    })

for raw_note, reason in REVIEW_DECISIONS.items():
    mapping_by_note[raw_note].update({
        "relation": "REVIEW",
        "evidence_source": "Candidate rule; existing project evidence insufficient",
        "reason": reason,
        "manual_review_required": True,
    })

for raw_note, reason in NOT_SAME_DECISIONS.items():
    mapping_by_note[raw_note].update({
        "relation": "NOT_SAME",
        "evidence_source": "v0.3 dictionary/evidence, Stage 20 decisions, or explicit audit exclusion",
        "reason": reason,
        "manual_review_required": False,
    })

mapping = pd.DataFrame([mapping_by_note[note] for note in notes])
MAP_COLUMNS = [
    "raw_note", "canonical_note", "relation",
    "evidence_source", "reason", "manual_review_required",
]
mapping = mapping[MAP_COLUMNS]

assert len(mapping) == len(vocab) == 2523
assert mapping["raw_note"].nunique() == 2523
assert set(mapping["relation"]) == {"CANONICAL", "SAME_CONCEPT", "NOT_SAME", "REVIEW"}
assert mapping.loc[mapping["relation"].eq("REVIEW"), "manual_review_required"].all()
assert not mapping.loc[~mapping["relation"].eq("REVIEW"), "manual_review_required"].any()
assert mapping[["raw_note", "canonical_note"]].ne("").all().all()

mapping.to_csv(MAP_PATH, index=False, encoding="utf-8-sig")
print("Saved:", MAP_PATH.relative_to(ROOT), "rows:", len(mapping))
display(mapping["relation"].value_counts().to_frame("raw_note_count"))


Saved: data\scent_knowledge\fragrantica_note_canonical_map_v1.csv rows: 2523


,raw_note_count
relation,
CANONICAL,2461
SAME_CONCEPT,48
NOT_SAME,7
REVIEW,7


## 5. 핵심 수치와 대표 통합 사례


In [5]:
original_note_count = int(vocab["note"].nunique())
canonical_note_count = int(mapping["canonical_note"].nunique())
same_concept_raw_count = int(mapping["relation"].eq("SAME_CONCEPT").sum())
review_count = int(mapping["relation"].eq("REVIEW").sum())

cluster_sizes = mapping.groupby("canonical_note")["raw_note"].nunique()
same_concept_cluster_count = int((cluster_sizes > 1).sum())

metrics = pd.DataFrame([
    {"metric": "original_unique_note_count", "value": original_note_count},
    {"metric": "canonical_unique_note_concept_count", "value": canonical_note_count},
    {"metric": "same_concept_raw_note_count", "value": same_concept_raw_count},
    {"metric": "same_concept_cluster_count", "value": same_concept_cluster_count},
    {"metric": "review_raw_note_count", "value": review_count},
])
display(metrics)

same_rows = (
    mapping.loc[mapping["relation"].eq("SAME_CONCEPT")]
    .merge(vocab, left_on="raw_note", right_on="note", how="left")
    .sort_values(["perfume_count", "raw_note"], ascending=[False, True])
)
display(same_rows[
    ["raw_note", "canonical_note", "perfume_count", "occurrence_count", "evidence_source"]
].head(15))

cluster_examples = (
    mapping.groupby("canonical_note")["raw_note"]
    .agg(list)
    .loc[lambda series: series.map(len) > 1]
)
display(cluster_examples.head(20).to_frame("raw_notes"))


,metric,value
0,original_unique_note_count,2523
1,canonical_unique_note_concept_count,2492
2,same_concept_raw_note_count,48
3,same_concept_cluster_count,30
4,review_raw_note_count,7


,raw_note,canonical_note,perfume_count,occurrence_count,evidence_source
0,Musk,Musk-Like,49244,49571,scent_term_dictionary_v0.3.csv; scent_term_evi...
1,Woody Notes,Woody,8278,8385,scent_term_dictionary_v0.3.csv; scent_term_evi...
2,Lily-of-the-Valley,Muguet,7339,7366,scent_term_dictionary_v0.3.csv; scent_term_evi...
3,Agarwood (Oud),Agarwood,5939,6134,scent_term_dictionary_v0.3.csv; scent_term_evi...
4,Black Currant,Blackcurrant,5560,5566,scent_term_dictionary_v0.3.csv; scent_term_evi...
5,Citruses,Citrus,4982,4993,scent_term_dictionary_v0.3.csv; scent_term_evi...
6,Green Notes,Green,4529,4538,scent_term_dictionary_v0.3.csv; scent_term_evi...
7,Floral Notes,Floral,4436,4505,scent_term_dictionary_v0.3.csv; scent_term_evi...
8,Oud,Agarwood,3780,3895,scent_term_dictionary_v0.3.csv; scent_term_evi...
9,Lily of the Valley,Muguet,3105,3118,scent_term_dictionary_v0.3.csv; scent_term_evi...


,raw_notes
canonical_note,
Agarwood,"[Agarwood (Oud), Oud, Agarwood]"
Ambrox Super,"[Ambrox Super, AMBROX® SUPER]"
Angel's Trumpet,"[Angels Trumpet, Angel's Trumpet]"
Blackcurrant,"[Black Currant, Blackcurrant]"
Candy Apple,"[Candy apple, Candy Apple]"
Cashmirwood,"[Cashmirwood, Cashmir wood]"
Cereal,"[Cereals, Cereal]"
Citrus,"[Citruses, Citrus]"
Clove,"[Cloves, Clove]"


## 6. Stage 20 strict 결과의 최소 재해석

Stage 20 Coverage 전체를 다시 계산하지 않는다. Stage 17 strict raw Note와 v0.3에서 추가된 HIGH SAME_CONCEPT raw Note의 합집합만 canonical 기준으로 중복 제거한다.


In [6]:
stage20_strict_raw = set(
    stage17.loc[stage17["final_auto_status"].eq("STRICT_MATCH"), "note"]
)
stage20_strict_raw.update(same_v03["term"])
assert len(stage20_strict_raw) == 185
assert stage20_strict_raw.issubset(vocabulary_set)

stage20_strict_canonical_count = int(
    mapping.loc[
        mapping["raw_note"].isin(stage20_strict_raw),
        "canonical_note",
    ].nunique()
)

stage20_reference = pd.DataFrame([
    {"metric": "Stage 20 strict matched raw Note", "value": len(stage20_strict_raw)},
    {"metric": "Canonical 기준 strict matched concept", "value": stage20_strict_canonical_count},
])
display(stage20_reference)

print(
    "Canonicalization은 raw Note 분모와 strict matched raw 개수를 concept 단위로 함께 줄이므로, "
    "기존 occurrence/perfume Coverage 결론을 뒤집는 근거가 아니다."
)


,metric,value
0,Stage 20 strict matched raw Note,185
1,Canonical 기준 strict matched concept,178


Canonicalization은 raw Note 분모와 strict matched raw 개수를 concept 단위로 함께 줄이므로, 기존 occurrence/perfume Coverage 결론을 뒤집는 근거가 아니다.


## 7. Annotation 영향 확인

Golden Set은 읽기만 하며 수정하지 않는다. 현재 Gold label에 SAME_CONCEPT alias raw Note가 직접 들어간 사례가 있는지 확인한다.


In [7]:
gold = pd.read_excel(
    PATHS["golden_set"],
    sheet_name="Golden Set",
    header=4,
    dtype=str,
    keep_default_na=False,
)
assert len(gold) == 200
assert "gold_scent_preference" in gold.columns

same_aliases = set(mapping.loc[mapping["relation"].eq("SAME_CONCEPT"), "raw_note"])
annotation_alias_hits = []
for row in gold.itertuples(index=False):
    labels = json.loads(row.gold_scent_preference)
    matched = sorted(set(labels) & same_aliases)
    if matched:
        annotation_alias_hits.append({
            "query_id": row.query_id,
            "query_text": row.query_text,
            "matched_same_concept_aliases": matched,
        })

annotation_alias_hits_df = pd.DataFrame(annotation_alias_hits)
print("현재 Golden Set의 SAME_CONCEPT alias label 포함 Query:", len(annotation_alias_hits_df))
if not annotation_alias_hits_df.empty:
    display(annotation_alias_hits_df)

vanilla_rows = gold.loc[
    gold["gold_scent_preference"].str.contains("Vanilla", regex=False)
]
display(vanilla_rows[["query_id", "query_text", "gold_scent_preference"]])
print("Vanille은 기존 근거 부족으로 REVIEW이며 이번 map에서 Vanilla로 자동 통합하지 않았다.")


현재 Golden Set의 SAME_CONCEPT alias label 포함 Query: 1


,query_id,query_text,matched_same_concept_aliases
0,UQ0110,남자가 쓰기 좋은 머스크 향수 추천해줘,[Musk]


,query_id,query_text,gold_scent_preference
4,UQ0005,난 바닐라 향기가 좋은데. 너무 단 건 싫어.,"[""Vanilla""]"


Vanille은 기존 근거 부족으로 REVIEW이며 이번 map에서 Vanilla로 자동 통합하지 않았다.


## 8. Summary 생성


In [8]:
important_examples = same_rows[
    ["raw_note", "canonical_note", "perfume_count"]
].head(10)

same_example_lines = "\n".join(
    f"- {row.raw_note} → {row.canonical_note} (perfume_count {int(row.perfume_count):,})"
    for row in important_examples.itertuples(index=False)
)
review_lines = "\n".join(
    f"- {raw}: REVIEW — {reason}"
    for raw, reason in REVIEW_DECISIONS.items()
)
not_same_lines = "\n".join(
    f"- {raw}: NOT_SAME — {reason}"
    for raw, reason in NOT_SAME_DECISIONS.items()
)

summary = f"""# Fragrantica Note Canonicalization Audit

## 결과 요약

- 원래 Note 수: **{original_note_count:,}**
- Canonical Note 수: **{canonical_note_count:,}**
- SAME_CONCEPT로 통합된 raw Note 수: **{same_concept_raw_count:,}**
- SAME_CONCEPT cluster 수: **{same_concept_cluster_count:,}**
- REVIEW 후보 수: **{review_count:,}**
- 가장 중요한 결론: Fragrantica 내부에는 확실한 중복 표현이 존재하지만 전체 vocabulary의 작은 일부다. v0.3의 검증된 alias와 명확한 표기/철자/문법 차이만 통합하고, Vanilla/Vanille처럼 기존 근거가 부족한 언어 변형은 REVIEW로 남겼다.

## 상세

### 1. 분석 목적

Fragrantica Note Vocabulary 내부에서 같은 향 개념이 표기, 철자, 문법 또는 이미 검증된 alias 때문에 여러 raw Note로 저장됐는지 확인하고, 추천/검색에서 재사용할 최소 canonical lookup을 만들었다. 새로운 향 지식이나 Coverage mapping은 추가하지 않았다.

### 2. 후보 탐색 방식

2,523개 전체 Note에 casefold, 공백/hyphen/punctuation 제거, note/notes 접미사 제거, 보수적 singular/plural key를 적용했다. 추가로 정규화 문자열 길이 차이 2 이하이고 SequenceMatcher 0.94 이상인 pair를 후보로 만들었다. 문자열 유사도는 후보 생성에만 사용했고 최종 판정은 명시적 rule/decision table로 제한했다.

### 3. 기존 Evidence 재사용

- analysis_outputs/17_fragrantica_note_vocabulary.csv
- analysis_outputs/17_note_ifra_matching.csv
- data/scent_knowledge/scent_term_dictionary_v0.3.csv
- data/scent_knowledge/scent_term_evidence_v0.3.csv
- analysis_outputs/20_external_note_validation_decisions.csv

v0.3의 HIGH SAME_CONCEPT를 그대로 우선 적용했다. perfumes.csv/jsonl, 외부 웹, LLM, embedding은 사용하지 않았다.

### 4. SAME_CONCEPT 주요 사례

{same_example_lines}

Lily-of-the-Valley와 Lily of the Valley는 기존 dsm-firmenich evidence에 따라 Muguet로, Black Currant는 기존 v0.3 canonical term인 Blackcurrant로 통합했다. Oakmoss/oak moss, Ylang-Ylang/Ylang Ylang 같은 내부 표기 차이는 기존 raw label 중 명확한 대표 표기로 통합했다.

### 5. REVIEW / NOT_SAME 주요 사례

{review_lines}

{not_same_lines}

Mint/Peppermint, Moss/Oakmoss, Orange Blossom/Neroli, Pine/Pine Needle, White Musk/Musk-Like는 FAMILY/RELATED 또는 별도 concept 가능성을 보존하고 통합하지 않았다.

### 6. Canonicalization 전후 Note Vocabulary 변화

- raw unique Note: **{original_note_count:,}**
- canonical unique concept: **{canonical_note_count:,}**
- 감소: **{original_note_count - canonical_note_count:,} concepts**
- SAME_CONCEPT raw mapping: **{same_concept_raw_count:,}**
- 실제 중복 cluster: **{same_concept_cluster_count:,}**

전체 2,523개 lookup을 저장했으며 변경 없는 Note는 raw_note=canonical_note, relation=CANONICAL이다.

### 7. Stage 20 결과 해석에 미치는 영향

- Stage 20 strict matched raw Note: **{len(stage20_strict_raw):,}**
- canonical 기준 strict matched concept: **{stage20_strict_canonical_count:,}**

Canonicalization은 vocabulary 분모와 strict matched raw Note를 concept 단위로 함께 중복 제거한다. 이는 Stage 20의 occurrence-weighted/perfume-level Coverage를 다시 최적화하거나 기존 결론을 뒤집는 분석이 아니다.

### 8. Annotation에 미치는 영향

현재 200개 Golden Set에서 SAME_CONCEPT alias raw label이 직접 사용된 Query는 **{len(annotation_alias_hits_df):,}개**였다. Vanilla label은 1개 Query에 존재하고 Vanille label은 없었다. 현재 파일을 수정하지 않았다.

**향 의미가 동일한 표현은 Annotation에서 Canonical Term 하나로 평가하고, 실제 Fragrantica 검색에서는 동일 Canonical Concept에 속한 raw Note를 모두 확장 검색한다.**

단, Vanille은 현재 REVIEW이므로 추가 근거 없이 Vanilla로 자동 정답화하지 않는다.

### 9. 이번 분석에서 검증된 것

- v0.3 HIGH SAME_CONCEPT와 명확한 내부 표기/철자/문법 중복을 안전하게 canonical lookup으로 만들 수 있다.
- canonical lookup은 모든 2,523개 raw Note를 손실 없이 보존하면서 검색 시 alias 확장을 가능하게 한다.
- 문자열 유사 후보 중 일부는 실제로 NOT_SAME 또는 REVIEW이며 자동 merge하면 안 된다.

### 10. 아직 검증되지 않은 것

- REVIEW {review_count:,}개의 의미 동일성
- 공식 근거가 없는 번역/언어 alias 전반
- FAMILY/RELATED를 Semantic Bridge feature로 쓰는 방식
- Canonicalization 이후 향수 ranking 또는 사용자 만족 변화

### 11. Semantic Bridge Pilot에 사용할 Canonicalization 원칙

1. 입력과 Annotation은 canonical_note 하나로 비교한다.
2. 실제 Fragrantica 검색은 canonical_note에 속한 모든 SAME_CONCEPT raw_note로 확장한다.
3. REVIEW와 NOT_SAME은 identity mapping으로 유지하고 자동 확장하지 않는다.
4. FAMILY/RELATED는 canonical equivalence가 아닌 별도 feature/evidence layer로만 다룬다.
5. 새로운 alias는 기존 공식 evidence 또는 명확한 표기 규칙이 생길 때만 map version을 올려 추가한다.
"""

SUMMARY_PATH.write_text(summary, encoding="utf-8")
print("Saved:", SUMMARY_PATH.relative_to(ROOT))
display(Markdown(summary))


Saved: analysis_outputs\21_fragrantica_note_canonicalization_summary.md


# Fragrantica Note Canonicalization Audit

## 결과 요약

- 원래 Note 수: **2,523**
- Canonical Note 수: **2,492**
- SAME_CONCEPT로 통합된 raw Note 수: **48**
- SAME_CONCEPT cluster 수: **30**
- REVIEW 후보 수: **7**
- 가장 중요한 결론: Fragrantica 내부에는 확실한 중복 표현이 존재하지만 전체 vocabulary의 작은 일부다. v0.3의 검증된 alias와 명확한 표기/철자/문법 차이만 통합하고, Vanilla/Vanille처럼 기존 근거가 부족한 언어 변형은 REVIEW로 남겼다.

## 상세

### 1. 분석 목적

Fragrantica Note Vocabulary 내부에서 같은 향 개념이 표기, 철자, 문법 또는 이미 검증된 alias 때문에 여러 raw Note로 저장됐는지 확인하고, 추천/검색에서 재사용할 최소 canonical lookup을 만들었다. 새로운 향 지식이나 Coverage mapping은 추가하지 않았다.

### 2. 후보 탐색 방식

2,523개 전체 Note에 casefold, 공백/hyphen/punctuation 제거, note/notes 접미사 제거, 보수적 singular/plural key를 적용했다. 추가로 정규화 문자열 길이 차이 2 이하이고 SequenceMatcher 0.94 이상인 pair를 후보로 만들었다. 문자열 유사도는 후보 생성에만 사용했고 최종 판정은 명시적 rule/decision table로 제한했다.

### 3. 기존 Evidence 재사용

- analysis_outputs/17_fragrantica_note_vocabulary.csv
- analysis_outputs/17_note_ifra_matching.csv
- data/scent_knowledge/scent_term_dictionary_v0.3.csv
- data/scent_knowledge/scent_term_evidence_v0.3.csv
- analysis_outputs/20_external_note_validation_decisions.csv

v0.3의 HIGH SAME_CONCEPT를 그대로 우선 적용했다. perfumes.csv/jsonl, 외부 웹, LLM, embedding은 사용하지 않았다.

### 4. SAME_CONCEPT 주요 사례

- Musk → Musk-Like (perfume_count 49,244)
- Woody Notes → Woody (perfume_count 8,278)
- Lily-of-the-Valley → Muguet (perfume_count 7,339)
- Agarwood (Oud) → Agarwood (perfume_count 5,939)
- Black Currant → Blackcurrant (perfume_count 5,560)
- Citruses → Citrus (perfume_count 4,982)
- Green Notes → Green (perfume_count 4,529)
- Floral Notes → Floral (perfume_count 4,436)
- Oud → Agarwood (perfume_count 3,780)
- Lily of the Valley → Muguet (perfume_count 3,105)

Lily-of-the-Valley와 Lily of the Valley는 기존 dsm-firmenich evidence에 따라 Muguet로, Black Currant는 기존 v0.3 canonical term인 Blackcurrant로 통합했다. Oakmoss/oak moss, Ylang-Ylang/Ylang Ylang 같은 내부 표기 차이는 기존 raw label 중 명확한 대표 표기로 통합했다.

### 5. REVIEW / NOT_SAME 주요 사례

- Vanille: REVIEW — Vanilla/Vanille is a plausible language variant, but existing v0.3 evidence explicitly left Vanille unresolved.
- Californian Orange: REVIEW — California/Californian wording is plausible but is not a pure spelling rule.
- Virginian Cedar: REVIEW — Virginia/Virginian wording is plausible but existing evidence does not confirm identity.
- Woodsy Notes: REVIEW — Existing v0.3 evidence explicitly did not equate Woodsy Notes with Woody.
- White Wood: REVIEW — White Wood/White Woods may be grammatical variants, but existing evidence is unresolved.
- Cep: REVIEW — Cep/Cepes may be a language/plural variant, but existing evidence is absent.
- Gaiac Wood: REVIEW — Gaiac/Guaiac may be a spelling or language variant, but existing project evidence does not explicitly confirm alias identity.

- Peppermint: NOT_SAME — Mint is broader and existing v0.3 evidence separates Mentha material forms.
- Moss: NOT_SAME — Moss is RELATED to Mossy and must not be collapsed with Oakmoss.
- Neroli: NOT_SAME — Orange Blossom and Neroli are distinct bitter-orange flower materials in Stage 20 evidence.
- Pine needles: NOT_SAME — Pine and Pine Needle material are not automatically identical concepts.
- White Musk: NOT_SAME — Existing v0.3 relation to Musk-Like is FAMILY, not SAME_CONCEPT.
- Water Notes: NOT_SAME — Existing v0.3 relation to Watery is RELATED, not SAME_CONCEPT.
- Bearberry: NOT_SAME — Barberry and Bearberry are only string-similar and are different labels.

Mint/Peppermint, Moss/Oakmoss, Orange Blossom/Neroli, Pine/Pine Needle, White Musk/Musk-Like는 FAMILY/RELATED 또는 별도 concept 가능성을 보존하고 통합하지 않았다.

### 6. Canonicalization 전후 Note Vocabulary 변화

- raw unique Note: **2,523**
- canonical unique concept: **2,492**
- 감소: **31 concepts**
- SAME_CONCEPT raw mapping: **48**
- 실제 중복 cluster: **30**

전체 2,523개 lookup을 저장했으며 변경 없는 Note는 raw_note=canonical_note, relation=CANONICAL이다.

### 7. Stage 20 결과 해석에 미치는 영향

- Stage 20 strict matched raw Note: **185**
- canonical 기준 strict matched concept: **178**

Canonicalization은 vocabulary 분모와 strict matched raw Note를 concept 단위로 함께 중복 제거한다. 이는 Stage 20의 occurrence-weighted/perfume-level Coverage를 다시 최적화하거나 기존 결론을 뒤집는 분석이 아니다.

### 8. Annotation에 미치는 영향

현재 200개 Golden Set에서 SAME_CONCEPT alias raw label이 직접 사용된 Query는 **1개**였다. Vanilla label은 1개 Query에 존재하고 Vanille label은 없었다. 현재 파일을 수정하지 않았다.

**향 의미가 동일한 표현은 Annotation에서 Canonical Term 하나로 평가하고, 실제 Fragrantica 검색에서는 동일 Canonical Concept에 속한 raw Note를 모두 확장 검색한다.**

단, Vanille은 현재 REVIEW이므로 추가 근거 없이 Vanilla로 자동 정답화하지 않는다.

### 9. 이번 분석에서 검증된 것

- v0.3 HIGH SAME_CONCEPT와 명확한 내부 표기/철자/문법 중복을 안전하게 canonical lookup으로 만들 수 있다.
- canonical lookup은 모든 2,523개 raw Note를 손실 없이 보존하면서 검색 시 alias 확장을 가능하게 한다.
- 문자열 유사 후보 중 일부는 실제로 NOT_SAME 또는 REVIEW이며 자동 merge하면 안 된다.

### 10. 아직 검증되지 않은 것

- REVIEW 7개의 의미 동일성
- 공식 근거가 없는 번역/언어 alias 전반
- FAMILY/RELATED를 Semantic Bridge feature로 쓰는 방식
- Canonicalization 이후 향수 ranking 또는 사용자 만족 변화

### 11. Semantic Bridge Pilot에 사용할 Canonicalization 원칙

1. 입력과 Annotation은 canonical_note 하나로 비교한다.
2. 실제 Fragrantica 검색은 canonical_note에 속한 모든 SAME_CONCEPT raw_note로 확장한다.
3. REVIEW와 NOT_SAME은 identity mapping으로 유지하고 자동 확장하지 않는다.
4. FAMILY/RELATED는 canonical equivalence가 아닌 별도 feature/evidence layer로만 다룬다.
5. 새로운 alias는 기존 공식 evidence 또는 명확한 표기 규칙이 생길 때만 map version을 올려 추가한다.


## 9. 종료 검증


In [9]:
saved_map = pd.read_csv(MAP_PATH, keep_default_na=False)
saved_summary = SUMMARY_PATH.read_text(encoding="utf-8")

assert saved_map.columns.tolist() == MAP_COLUMNS
assert len(saved_map) == 2523
assert saved_map["raw_note"].nunique() == 2523
assert saved_map["canonical_note"].nunique() == canonical_note_count
assert int(saved_map["relation"].eq("SAME_CONCEPT").sum()) == same_concept_raw_count
assert int(saved_map["relation"].eq("REVIEW").sum()) == review_count
assert "Semantic Bridge Pilot에 사용할 Canonicalization 원칙" in saved_summary

print("검증 완료")
print("- Notebook 질문: Fragrantica 내부 duplicate Note canonicalization")
print("- 출력 파일:", MAP_PATH.relative_to(ROOT))
print("- 출력 파일:", SUMMARY_PATH.relative_to(ROOT))
print("- 외부 웹/API/LLM/embedding/ranking 실행: 0건")
print("- 기존 v0.3 / Golden Set 수정: 0건")


검증 완료
- Notebook 질문: Fragrantica 내부 duplicate Note canonicalization
- 출력 파일: data\scent_knowledge\fragrantica_note_canonical_map_v1.csv
- 출력 파일: analysis_outputs\21_fragrantica_note_canonicalization_summary.md
- 외부 웹/API/LLM/embedding/ranking 실행: 0건
- 기존 v0.3 / Golden Set 수정: 0건
